# 11 - Prepare external CXR image-report pairs

The CXR release has two explicit partitions:

- training: `/data/liangz2/openi/multi_kg/train`
- test: `/data/liangz2/openi/multi_kg/test`

The provided test partition remains untouched. The provided training partition is divided patient-wise into training and validation subsets. If a patient occurs in both source folders, test takes precedence and all records for that patient are assigned to the test split. JSON field lookup is case-insensitive; the primary report field is `Caption`, with `FINDINGS` and `IMPRESSION` as fallbacks.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re

import pandas as pd
from tqdm.auto import tqdm

ARTIFACT_ROOT = Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
CXR_TRAIN_ROOT = Path(os.environ.get("MIMIC_CXR_TRAIN_ROOT", config.get("external_cxr_root", "/data/liangz2/openi/multi_kg/train")))
CXR_TEST_ROOT = Path(os.environ.get("MIMIC_CXR_TEST_ROOT", config.get("external_cxr_test_root", "/data/liangz2/openi/multi_kg/test")))
OUT = ARTIFACT_ROOT / "external" / "cxr"
OUT.mkdir(parents=True, exist_ok=True)
ROOTS = {"provided_train": CXR_TRAIN_ROOT, "provided_test": CXR_TEST_ROOT}
for name, root in ROOTS.items():
    if not root.is_dir():
        raise FileNotFoundError(f"Missing {name} CXR directory: {root}")

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def discover_pairs(root):
    images = sorted(path for path in root.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES)
    json_files = sorted(path for path in root.rglob("*") if path.suffix.lower() == ".json")
    json_by_stem = {}
    duplicate_stems = []
    for path in json_files:
        key = path.relative_to(root).with_suffix("").as_posix().lower()
        if key in json_by_stem:
            duplicate_stems.append(key)
        else:
            json_by_stem[key] = path
    pairs = []
    for image_path in images:
        key = image_path.relative_to(root).with_suffix("").as_posix().lower()
        if key in json_by_stem:
            pairs.append((image_path, json_by_stem[key]))
    return pairs, {"images": len(images), "json_files": len(json_files),
                   "pairs": len(pairs), "unpaired_images": len(images) - len(pairs),
                   "duplicate_json_stems": duplicate_stems}

discovered = {}
discovery_stats = {}
for partition, root in ROOTS.items():
    discovered[partition], discovery_stats[partition] = discover_pairs(root)
    if not discovered[partition]:
        raise RuntimeError(f"No same-stem image/JSON pairs under {root}")
print(json.dumps({name: {**stats, "root": str(ROOTS[name])}
                  for name, stats in discovery_stats.items()}, indent=2))


In [ ]:
def casefold_map(meta):
    return {str(key).casefold(): value for key, value in meta.items()}

def first(meta, keys):
    folded = casefold_map(meta)
    for key in keys:
        if key.casefold() in folded:
            value = folded[key.casefold()]
            if value is not None and not (isinstance(value, str) and not value.strip()):
                return value
    return None

def text_value(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return " ".join(value.split())
    if isinstance(value, (list, tuple)):
        return " ".join(part for part in (text_value(item) for item in value) if part)
    if isinstance(value, dict):
        folded = casefold_map(value)
        ordered = [folded.get(key) for key in ["findings", "impression", "text", "caption"] if key in folded]
        return " ".join(part for part in (text_value(item) for item in ordered) if part)
    return str(value).strip()

def report_text_and_source(meta):
    folded = casefold_map(meta)
    for key in ["caption", "report_text", "report", "text"]:
        if key in folded and text_value(folded[key]):
            return text_value(folded[key]), key
    parts = [text_value(folded.get(key)) for key in ["findings", "impression"]]
    text = " ".join(part for part in parts if part)
    return text, "findings+impression" if text else ""

def integer_or_none(value):
    if value is None or (isinstance(value, str) and not value.strip()):
        return None
    match = re.search(r"(\d+)$", str(value).strip())
    return int(match.group(1)) if match else None

def normalized_labels(value):
    if value is None:
        return []
    if isinstance(value, dict):
        return sorted(str(key) for key, present in value.items()
                      if present not in (None, False, 0, 0.0, "", "0", "false", "False"))
    if isinstance(value, str):
        return [part.strip() for part in re.split(r"[,;|]", value) if part.strip()]
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    return [str(value).strip()]

rows, bad_json = [], []
blank_reports = {name: 0 for name in ROOTS}
for partition, pairs in discovered.items():
    root = ROOTS[partition]
    for image_path, json_path in tqdm(pairs, desc=partition):
        try:
            meta = json.loads(json_path.read_text())
            if not isinstance(meta, dict):
                raise TypeError(f"Expected a JSON object, got {type(meta).__name__}")
        except Exception as exc:
            bad_json.append({"partition": partition, "path": str(json_path), "error": repr(exc)})
            continue
        text, report_source = report_text_and_source(meta)
        if not text:
            blank_reports[partition] += 1
            continue
        relative = image_path.relative_to(root).with_suffix("").as_posix()
        study_id = integer_or_none(first(meta, ["study_id", "dicom_study_id"]))
        rows.append({
            "sample_id": "cxr_" + relative.replace("/", "__"),
            "image_path": str(image_path), "json_path": str(json_path),
            "report_text": text, "report_source": report_source,
            "labels": normalized_labels(first(meta, ["labels", "chexpert_labels", "positive_labels"])),
            "normal": first(meta, ["normal"]), "view_position": first(meta, ["view_position"]),
            "subject_id": integer_or_none(first(meta, ["subject_id", "patient_id", "patientId"])),
            "hadm_id": integer_or_none(first(meta, ["hadm_id", "admission_id", "encounter_id"])),
            "study_id": study_id, "source_partition": partition,
        })

frame = pd.DataFrame(rows)
if frame.empty:
    raise RuntimeError(f"All JSON reports were blank or invalid; blank={blank_reports}, bad_json={bad_json[:10]}")
if not frame.sample_id.is_unique:
    duplicates = frame.loc[frame.sample_id.duplicated(False), ["sample_id", "source_partition", "json_path"]]
    raise ValueError(f"Duplicate CXR sample IDs across partitions: {duplicates.head(20).to_dict('records')}")
print({"usable_pairs": len(frame), "bad_json": len(bad_json), "blank_reports": blank_reports,
       "report_sources": frame.report_source.value_counts().to_dict(),
       "with_subject_id": int(frame.subject_id.notna().sum()),
       "with_study_id": int(frame.study_id.notna().sum())})
display(frame[["sample_id", "subject_id", "study_id", "labels", "report_source", "source_partition"]].head())


In [ ]:
def stable_bucket(value):
    return int(hashlib.sha1(str(value).encode()).hexdigest()[:8], 16) % 100

VAL_PERCENT = int(os.environ.get("CXR_VALIDATION_PERCENT", "10"))
if not 1 <= VAL_PERCENT <= 50:
    raise ValueError("CXR_VALIDATION_PERCENT must be between 1 and 50")

frame["split"] = "test"
provided_train = frame.source_partition.eq("provided_train")
split_keys = []
for row in frame.loc[provided_train].itertuples():
    if pd.notna(row.subject_id):
        split_keys.append(str(int(row.subject_id)))
    elif pd.notna(row.hadm_id):
        split_keys.append(f"admission:{int(row.hadm_id)}")
    else:
        split_keys.append(f"sample:{row.sample_id}")
buckets = pd.Series(split_keys, index=frame.index[provided_train]).map(stable_bucket)
frame.loc[provided_train, "split"] = buckets.map(
    lambda value: "val" if value >= 100 - VAL_PERCENT else "train")

# Enforce patient-disjoint model splits. The source folders themselves can overlap;
# in that case, the supplied test partition takes precedence for the whole patient.
train_subjects = set(frame.loc[frame.source_partition.eq("provided_train"), "subject_id"].dropna().astype("int64"))
test_subjects = set(frame.loc[frame.source_partition.eq("provided_test"), "subject_id"].dropna().astype("int64"))
source_overlap = train_subjects & test_subjects
overlap_train_rows = provided_train & frame.subject_id.isin(source_overlap)
reassigned_overlap_rows = int(overlap_train_rows.sum())
frame.loc[overlap_train_rows, "split"] = "test"
known_subjects = frame.loc[frame.subject_id.notna()]
split_counts = known_subjects.groupby("subject_id")["split"].nunique()
leaking = split_counts[split_counts > 1]
if not leaking.empty:
    raise AssertionError(f"CXR subjects assigned to multiple splits: {leaking.head(20).to_dict()}")

split_basis = (
    "provided test partition with patient-level test precedence for source-folder overlaps; "
    "patient-hash train/validation split for remaining provided-training patients")
print(frame.groupby("split").agg(rows=("sample_id", "size"), subjects=("subject_id", "nunique")))
print({"validation_percent_of_provided_train": VAL_PERCENT,
       "source_overlap_subjects": len(source_overlap),
       "provided_train_rows_reassigned_to_test": reassigned_overlap_rows,
       "source_overlap_examples": [int(value) for value in sorted(source_overlap)[:20]],
       "split_basis": split_basis})


In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(value, "item"):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass
    return value

output = OUT / "manifest.jsonl"
with output.open("w") as stream:
    for row in frame.to_dict("records"):
        stream.write(json.dumps(json_safe(row)) + "\n")
summary = {
    "rows": len(frame),
    "splits": {str(key): int(value) for key, value in frame.split.value_counts().items()},
    "source_partitions": {str(key): int(value) for key, value in frame.source_partition.value_counts().items()},
    "split_basis": split_basis, "validation_percent": VAL_PERCENT,
    "source_overlap_subjects": len(source_overlap),
    "provided_train_rows_reassigned_to_test": reassigned_overlap_rows,
    "source_overlap_examples": [int(value) for value in sorted(source_overlap)[:20]],
    "with_subject_id": int(frame.subject_id.notna().sum()),
    "with_hadm_id": int(frame.hadm_id.notna().sum()),
    "with_study_id": int(frame.study_id.notna().sum()),
    "report_sources": {str(key): int(value) for key, value in frame.report_source.value_counts().items()},
    "blank_reports": blank_reports,
    "discovery": discovery_stats,
    "bad_json": bad_json[:20],
}
(OUT / "manifest_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Saved:", output)
